We formulate FPL squad selection as a Mixed-Integer Linear Program. The solver maximises expected points over a rolling horizon of gameweeks, subject to all FPL game rules: budget, squad composition, and transfer costs.

In [1]:
formulation = """\
Objective
  max  Z = \u03a3_w  \u03b3^w \u00d7 (SquadPoints_w \u2212 TransferCost_w)

  SquadPoints_w = \u03a3_i  xp_iw \u00d7 xi_iw  +  xp_cw \u00d7 captain_w   (captain scores double)
  TransferCost_w = transfer_cost \u00d7 max(0, transfers_w \u2212 free_transfers_w)

Subject to
  \u03a3_i squad_iw = 15                                    (squad size)
  \u03a3_i xi_iw   = 11                                     (starting XI)
  squad composition: 2 GKP, 5 DEF, 5 MID, 3 FWD        (positional limits)
  \u03a3_{i \u2208 team_t} squad_iw \u2264 3    \u2200t                      (max 3 per club)
  \u03a3_i cost_i \u00d7 squad_iw \u2264 budget_w                       (budget)
  xi_iw \u2264 squad_iw                                      (can only start squad members)
  captain_w \u2208 {xi_iw}                                   (captain must be in XI)

  \u03b3 = 0.95  (decay),  transfer_cost = 4 pts,  horizon = 5 GWs
"""
print(formulation)

Objective
  max  Z = Σ_w  γ^w × (SquadPoints_w − TransferCost_w)

  SquadPoints_w = Σ_i  xp_iw × xi_iw  +  xp_cw × captain_w   (captain scores double)
  TransferCost_w = transfer_cost × max(0, transfers_w − free_transfers_w)

Subject to
  Σ_i squad_iw = 15                                    (squad size)
  Σ_i xi_iw   = 11                                     (starting XI)
  squad composition: 2 GKP, 5 DEF, 5 MID, 3 FWD        (positional limits)
  Σ_{i ∈ team_t} squad_iw ≤ 3    ∀t                      (max 3 per club)
  Σ_i cost_i × squad_iw ≤ budget_w                       (budget)
  xi_iw ≤ squad_iw                                      (can only start squad members)
  captain_w ∈ {xi_iw}                                   (captain must be in XI)

  γ = 0.95  (decay),  transfer_cost = 4 pts,  horizon = 5 GWs


The decision variables are all binary: is player *i* in the squad in week *w*? In the starting XI? Captain? The solver explores the full combinatorial space and returns a provably optimal plan — which transfers to make, who to start, who to captain — across the entire horizon.

In [2]:
import pandas as pd

backtest = pd.read_csv("data/backtests/comparison_20260215_233723/backtest_summary.csv")
backtest[["predictor", "point_spearman", "point_rmse", "point_mae", "point_top_n_points"]]

Predictor,Spearman ρ,RMSE,MAE,Top-N pts
OpenFPL (self-trained),0.763,1.41,0.85,3249
LightGBM,0.669,1.99,1.02,984
Heuristic,0.653,2.24,1.15,965
OpenFPL (pre-trained),0.650,2.00,1.15,983


OpenFPL dominates on prediction quality: Spearman 0.76 vs. 0.67 for LightGBM. But better predictions don't automatically mean more FPL points.

In [3]:
strategy = pd.read_csv("data/backtests/comparison_20260215_233723/strategy_summary.csv")
strategy[["predictor", "total_raw_points", "total_hit_points", "total_net_points",
          "total_transfers", "total_paid_transfers"]].sort_values("total_net_points", ascending=False)

Predictor,Raw pts,Hit pts,Net pts,Transfers,Paid transfers
LightGBM,1070,4,1066,25,1
Heuristic,1141,188,953,71,47
OpenFPL (self-trained),979,124,855,55,31
OpenFPL (pre-trained),841,0,841,24,0


The heuristic predictor produces high expected-point predictions that tempt the solver into excessive transfers — 47 paid transfers costing 188 points in hits. LightGBM's more conservative predictions lead to only 1 paid transfer across 24 gameweeks, finishing with the highest net score despite ranking second in raw prediction accuracy.